# 財務資訊管理期中小組作業：MACD 與 RSI 技術指標交易策略

本 notebook 以 Google Colab 為開發環境，使用 `yfinance` 抓取股票日資料，完成資料處理、技術指標計算、交易訊號設計、回測績效分析、圖表視覺化，以及策略結果解釋。

策略主軸採用 **MACD 趨勢動能指標** 搭配 **RSI 強弱指標**。MACD 用來判斷趨勢轉折，RSI 用來過濾動能不足或偏弱的訊號，避免只因短期交叉就過度交易。

## 1. 安裝與匯入套件

若在 Colab 執行，第一次執行時可能需要安裝 `yfinance`。

In [ ]:
!pip -q install yfinance

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

: 

## 2. 研究設定與資料抓取

本作業以台積電 ADR (`TSM`) 作為主要分析標的，期間設定為 2018-01-01 至 2026-05-06。選擇 ADR 的原因是 Yahoo Finance 資料穩定，且成交量、調整後價格等欄位較完整。

加分分析會另外比較 Apple (`AAPL`)、NVIDIA (`NVDA`) 與 SPY ETF (`SPY`)。

In [ ]:
MAIN_TICKER = "TSM"
COMPARE_TICKERS = ["TSM", "AAPL", "NVDA", "SPY"]
START_DATE = "2018-01-01"
END_DATE = "2026-05-06"
CACHE_DIR = "price_cache"

def download_price_data(ticker, start=START_DATE, end=END_DATE, use_cache=True):
    import os
    os.makedirs(CACHE_DIR, exist_ok=True)
    cache_path = os.path.join(CACHE_DIR, f"{ticker}_{start}_{end}.csv")

    if use_cache and os.path.exists(cache_path):
        df = pd.read_csv(cache_path)
        print(f"???????{cache_path}")
    else:
        df = yf.download(ticker, start=start, end=end, auto_adjust=False, progress=False, threads=False)
        if df.empty:
            raise ValueError(
                f"???? {ticker} ??????? Yahoo Finance ?????"
                "??????????????????????"
            )
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)
        df = df.reset_index()
        df.to_csv(cache_path, index=False)
        print(f"?????????{cache_path}")

    df["Date"] = pd.to_datetime(df["Date"])
    df = df.sort_values("Date").drop_duplicates("Date")
    df = df.set_index("Date")
    price_col = "Adj Close" if "Adj Close" in df.columns else "Close"
    df["Price"] = df[price_col]
    df = df.dropna(subset=["Open", "High", "Low", "Close", "Price", "Volume"])
    return df

raw = download_price_data(MAIN_TICKER)
display(raw.head())
display(raw.tail())
print(f"?????{len(raw):,}")
print(f"?????{raw.index.min().date()} ? {raw.index.max().date()}")

## 3. 資料預先處理

預處理包含：

1. 將日期轉為 `datetime` 並設為索引。
2. 依日期排序並移除重複日期。
3. 使用調整後收盤價 `Adj Close` 作為回測價格，以反映股利與拆股影響。
4. 移除必要欄位缺值。
5. 計算每日報酬率，作為後續策略績效分析基礎。

In [ ]:
def preprocess_data(df):
    out = df.copy()
    out["Return"] = out["Price"].pct_change()
    out["Log_Return"] = np.log(out["Price"] / out["Price"].shift(1))
    out["MA_200"] = out["Price"].rolling(200).mean()
    return out.dropna(subset=["Return"])

data = preprocess_data(raw)
missing_summary = data[["Open", "High", "Low", "Close", "Price", "Volume", "Return"]].isna().sum()
display(missing_summary.to_frame("缺值數量"))
display(data[["Price", "Return", "Volume"]].describe())

## 4. 技術指標計算

### MACD

MACD 使用 12 日與 26 日指數移動平均線（EMA）的差：

$$MACD_t = EMA_{12,t} - EMA_{26,t}$$

訊號線為 MACD 的 9 日 EMA：

$$Signal_t = EMA_9(MACD_t)$$

柱狀體為 MACD 與 Signal 的差，可觀察動能擴張或收斂。

### RSI

RSI 使用 14 日平均漲幅與平均跌幅計算：

$$RSI = 100 - \frac{100}{1 + RS}$$

其中 $RS$ 為平均上漲幅度除以平均下跌幅度。RSI 高於 50 通常代表偏多動能較強，低於 50 則偏弱。

In [ ]:
def add_indicators(df, fast=12, slow=26, signal=9, rsi_window=14):
    out = df.copy()
    out["EMA_fast"] = out["Price"].ewm(span=fast, adjust=False).mean()
    out["EMA_slow"] = out["Price"].ewm(span=slow, adjust=False).mean()
    out["MACD"] = out["EMA_fast"] - out["EMA_slow"]
    out["MACD_signal"] = out["MACD"].ewm(span=signal, adjust=False).mean()
    out["MACD_hist"] = out["MACD"] - out["MACD_signal"]

    delta = out["Price"].diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.ewm(alpha=1 / rsi_window, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1 / rsi_window, adjust=False).mean()
    rs = avg_gain / avg_loss.replace(0, np.nan)
    out["RSI"] = 100 - (100 / (1 + rs))
    return out

data = add_indicators(data).dropna()
display(data[["Price", "MACD", "MACD_signal", "MACD_hist", "RSI"]].tail())

## 5. 交易策略設計

本策略採取「只做多、不放空」的設定，較符合一般投資人操作習慣。

交易規則：

1. **進場訊號**：MACD 由下往上突破 Signal，且 RSI 大於 50。
2. **出場訊號**：MACD 由上往下跌破 Signal，或 RSI 小於 45。
3. **延遲進場**：訊號產生於收盤後，因此隔天才持有部位，避免使用未來資料。
4. **交易成本**：每次部位變動扣除 0.1425% 成本，近似台股常見手續費率；ADR 實際成本可能不同，此處作為保守假設。

In [ ]:
def add_strategy(df, transaction_cost=0.001425):
    out = df.copy()
    macd_cross_up = (out["MACD"] > out["MACD_signal"]) & (out["MACD"].shift(1) <= out["MACD_signal"].shift(1))
    macd_cross_down = (out["MACD"] < out["MACD_signal"]) & (out["MACD"].shift(1) >= out["MACD_signal"].shift(1))
    out["Buy_Signal"] = macd_cross_up & (out["RSI"] > 50)
    out["Sell_Signal"] = macd_cross_down | (out["RSI"] < 45)

    position = []
    holding = 0
    for buy, sell in zip(out["Buy_Signal"], out["Sell_Signal"]):
        if holding == 0 and buy:
            holding = 1
        elif holding == 1 and sell:
            holding = 0
        position.append(holding)

    out["Signal_Position"] = position
    out["Position"] = out["Signal_Position"].shift(1).fillna(0)
    out["Trade"] = out["Position"].diff().abs().fillna(0)
    out["Strategy_Return_Gross"] = out["Position"] * out["Return"]
    out["Transaction_Cost"] = out["Trade"] * transaction_cost
    out["Strategy_Return"] = out["Strategy_Return_Gross"] - out["Transaction_Cost"]
    out["BuyHold_Cum"] = (1 + out["Return"]).cumprod()
    out["Strategy_Cum"] = (1 + out["Strategy_Return"]).cumprod()
    return out

result = add_strategy(data)
display(result[["Price", "MACD", "MACD_signal", "RSI", "Buy_Signal", "Sell_Signal", "Position", "Strategy_Return"]].tail())
print(f"買進訊號次數：{int(result['Buy_Signal'].sum())}")
print(f"賣出訊號次數：{int(result['Sell_Signal'].sum())}")
print(f"實際交易次數：{int(result['Trade'].sum())}")

## 6. 績效衡量函數

使用累積報酬率、年化報酬率、年化波動率、Sharpe Ratio、最大回撤與勝率等指標評估策略。

In [ ]:
def max_drawdown(cumulative_return):
    running_max = cumulative_return.cummax()
    drawdown = cumulative_return / running_max - 1
    return drawdown.min(), drawdown

def performance_metrics(df, return_col, cumulative_col):
    daily_ret = df[return_col].dropna()
    total_return = df[cumulative_col].iloc[-1] - 1
    years = len(daily_ret) / 252
    annual_return = (1 + total_return) ** (1 / years) - 1
    annual_vol = daily_ret.std() * np.sqrt(252)
    sharpe = annual_return / annual_vol if annual_vol != 0 else np.nan
    mdd, _ = max_drawdown(df[cumulative_col])
    win_rate = (daily_ret > 0).mean()
    return {
        "累積報酬率": total_return,
        "年化報酬率": annual_return,
        "年化波動率": annual_vol,
        "Sharpe Ratio": sharpe,
        "最大回撤": mdd,
        "日勝率": win_rate,
    }

metrics = pd.DataFrame({
    "Buy and Hold": performance_metrics(result, "Return", "BuyHold_Cum"),
    "MACD + RSI Strategy": performance_metrics(result, "Strategy_Return", "Strategy_Cum"),
}).T

display(metrics.style.format("{:.2%}", subset=["累積報酬率", "年化報酬率", "年化波動率", "最大回撤", "日勝率"]).format("{:.2f}", subset=["Sharpe Ratio"]))

## 7. 技術分析圖表

下圖同時呈現價格、買賣點、MACD 與 RSI。買進點以綠色三角形表示，賣出點以紅色倒三角形表示。

In [ ]:
def plot_strategy(df, ticker):
    fig, axes = plt.subplots(4, 1, figsize=(15, 12), sharex=True, gridspec_kw={"height_ratios": [3, 1.5, 1.5, 2]})

    buy_points = df[df["Buy_Signal"]]
    sell_points = df[df["Sell_Signal"]]

    axes[0].plot(df.index, df["Price"], label="Adjusted Price", color="#1f77b4")
    axes[0].plot(df.index, df["MA_200"], label="200-day MA", color="#ff7f0e", alpha=0.8)
    axes[0].scatter(buy_points.index, buy_points["Price"], marker="^", color="green", label="Buy Signal", s=55)
    axes[0].scatter(sell_points.index, sell_points["Price"], marker="v", color="red", label="Sell Signal", s=45)
    axes[0].set_title(f"{ticker} Price and Trading Signals")
    axes[0].set_ylabel("Price")
    axes[0].legend(loc="upper left")

    axes[1].plot(df.index, df["MACD"], label="MACD", color="#2ca02c")
    axes[1].plot(df.index, df["MACD_signal"], label="Signal", color="#d62728")
    axes[1].bar(df.index, df["MACD_hist"], label="Histogram", color=np.where(df["MACD_hist"] >= 0, "#8fd19e", "#f2a6a6"), width=1.0)
    axes[1].axhline(0, color="black", linewidth=0.8)
    axes[1].set_title("MACD Indicator")
    axes[1].set_ylabel("MACD")
    axes[1].legend(loc="upper left")

    axes[2].plot(df.index, df["RSI"], label="RSI", color="#9467bd")
    axes[2].axhline(70, color="red", linestyle="--", linewidth=0.9, label="Overbought 70")
    axes[2].axhline(50, color="gray", linestyle="--", linewidth=0.9, label="Momentum 50")
    axes[2].axhline(30, color="green", linestyle="--", linewidth=0.9, label="Oversold 30")
    axes[2].set_title("RSI Indicator")
    axes[2].set_ylabel("RSI")
    axes[2].set_ylim(0, 100)
    axes[2].legend(loc="upper left")

    axes[3].plot(df.index, df["BuyHold_Cum"], label="Buy and Hold", color="#7f7f7f")
    axes[3].plot(df.index, df["Strategy_Cum"], label="MACD + RSI Strategy", color="#17becf")
    axes[3].set_title("Cumulative Return Comparison")
    axes[3].set_ylabel("Growth of $1")
    axes[3].legend(loc="upper left")

    plt.xlabel("Date")
    plt.tight_layout()
    plt.show()

plot_strategy(result, MAIN_TICKER)

## 8. 回撤分析與交易統計

最大回撤可以觀察策略在最不利期間可能承受的虧損幅度。若策略報酬率沒有明顯提高，但最大回撤較低，仍代表策略有風險控制價值。

In [ ]:
buyhold_mdd, buyhold_dd = max_drawdown(result["BuyHold_Cum"])
strategy_mdd, strategy_dd = max_drawdown(result["Strategy_Cum"])

plt.figure(figsize=(15, 5))
plt.plot(buyhold_dd.index, buyhold_dd, label="Buy and Hold Drawdown", color="#7f7f7f")
plt.plot(strategy_dd.index, strategy_dd, label="Strategy Drawdown", color="#17becf")
plt.title("Drawdown Comparison")
plt.ylabel("Drawdown")
plt.xlabel("Date")
plt.legend()
plt.show()

trade_entries = result[(result["Position"].diff() == 1)].copy()
trade_exits = result[(result["Position"].diff() == -1)].copy()
if len(trade_exits) < len(trade_entries):
    trade_exits = pd.concat([trade_exits, result.tail(1)])

trades = []
for entry_date, exit_date in zip(trade_entries.index, trade_exits.index):
    entry_price = result.loc[entry_date, "Price"]
    exit_price = result.loc[exit_date, "Price"]
    trades.append({
        "進場日": entry_date,
        "出場日": exit_date,
        "持有天數": (exit_date - entry_date).days,
        "進場價": entry_price,
        "出場價": exit_price,
        "單筆報酬率": exit_price / entry_price - 1,
    })

trade_table = pd.DataFrame(trades)
display(trade_table.tail(10))

if not trade_table.empty:
    trade_summary = pd.Series({
        "交易筆數": len(trade_table),
        "平均單筆報酬率": trade_table["單筆報酬率"].mean(),
        "單筆勝率": (trade_table["單筆報酬率"] > 0).mean(),
        "平均持有天數": trade_table["持有天數"].mean(),
        "最佳單筆報酬率": trade_table["單筆報酬率"].max(),
        "最差單筆報酬率": trade_table["單筆報酬率"].min(),
    })
    display(trade_summary.to_frame("數值"))

## 9. 加分分析一：多檔股票比較

同一套策略不一定適用所有標的。趨勢明顯、波動較大的股票較可能讓 MACD 類策略發揮；橫盤震盪或假突破很多的股票，則容易產生來回交易成本。

In [ ]:
comparison_rows = []
comparison_curves = {}

for ticker in COMPARE_TICKERS:
    df = download_price_data(ticker)
    df = preprocess_data(df)
    df = add_indicators(df).dropna()
    df = add_strategy(df)
    comparison_curves[ticker] = df[["Strategy_Cum", "BuyHold_Cum"]]
    row = performance_metrics(df, "Strategy_Return", "Strategy_Cum")
    row["Ticker"] = ticker
    row["交易次數"] = int(df["Trade"].sum())
    comparison_rows.append(row)

comparison = pd.DataFrame(comparison_rows).set_index("Ticker")
display(comparison.style.format("{:.2%}", subset=["累積報酬率", "年化報酬率", "年化波動率", "最大回撤", "日勝率"]).format("{:.2f}", subset=["Sharpe Ratio"]))

plt.figure(figsize=(15, 6))
for ticker, curve in comparison_curves.items():
    plt.plot(curve.index, curve["Strategy_Cum"], label=f"{ticker} Strategy")
plt.title("MACD + RSI Strategy Across Different Assets")
plt.ylabel("Growth of $1")
plt.xlabel("Date")
plt.legend()
plt.show()

## 10. 加分分析二：牛市與熊市分段比較

此處使用價格是否高於 200 日均線定義市場狀態：

1. 價格高於 200 日均線：偏牛市或上升趨勢。
2. 價格低於 200 日均線：偏熊市或下降趨勢。

這種分類不代表真正總體經濟牛熊市，但能檢查策略在不同趨勢環境中的表現差異。

In [ ]:
regime = result.dropna(subset=["MA_200"]).copy()
regime["Market_Regime"] = np.where(regime["Price"] >= regime["MA_200"], "價格高於200日均線", "價格低於200日均線")

regime_summary = regime.groupby("Market_Regime").agg(
    策略平均日報酬=("Strategy_Return", "mean"),
    買進持有平均日報酬=("Return", "mean"),
    策略日波動=("Strategy_Return", "std"),
    策略日勝率=("Strategy_Return", lambda x: (x > 0).mean()),
    樣本天數=("Strategy_Return", "count"),
)
regime_summary["策略年化報酬估計"] = (1 + regime_summary["策略平均日報酬"]) ** 252 - 1
display(regime_summary)

plt.figure(figsize=(10, 5))
plt.bar(regime_summary.index, regime_summary["策略年化報酬估計"], color=["#4c78a8", "#f58518"])
plt.title("Estimated Annual Strategy Return by Market Regime")
plt.ylabel("Estimated Annual Return")
plt.xticks(rotation=0)
plt.show()

## 11. 策略結果解釋

MACD 本質上是趨勢追蹤指標，所以當標的出現明確上升趨勢時，策略容易透過黃金交叉進場並捕捉一段趨勢報酬。RSI 的加入則是為了確認多方動能，避免 MACD 在弱勢反彈時過早進場。

若策略績效優於買進持有，通常代表策略成功避開部分下跌區段，或在大跌前透過 MACD 死亡交叉與 RSI 轉弱訊號出場。若策略績效落後，常見原因是標的長期多頭時，頻繁出場會錯過後續反彈；此外，盤整期間 MACD 容易反覆交叉，造成交易成本增加。

從多檔股票比較可以觀察到，同一組參數對不同股票的效果可能差異很大。這代表技術指標不是固定獲利公式，而是需要配合標的特性、市場環境與交易成本評估。

## 12. 生成式 AI 應用與反思

本作業有使用生成式 AI 協助規劃 notebook 架構與程式碼撰寫。為了讓 AI 產生較高品質的結果，我採用以下溝通方式：

1. 先提供評分規準，讓 AI 明確知道 notebook 必須包含資料抓取、資料處理、指標計算、交易策略、圖表與結果分析。
2. 要求程式碼分段撰寫，避免全部邏輯塞在同一個 cell，讓後續除錯與報告呈現更清楚。
3. 要求策略避免未來函數問題，因此交易部位使用 `shift(1)`，代表訊號產生後隔日才進場。
4. 要求加入交易成本與多檔股票比較，避免回測結果過度樂觀，也讓分析更接近真實投資情境。

使用 AI 的過程中，仍需要人工檢查指標公式、交易邏輯與圖表是否合理。特別是技術指標回測很容易不小心使用未來資料，例如用同一天收盤產生訊號後又立刻用同一天報酬計算績效，因此必須檢查部位是否有延遲。未來可以進一步加入參數敏感度分析、停損停利、樣本外測試或 walk-forward validation，提高策略檢驗的可信度。

## 13. 結論

本 notebook 完成 MACD 與 RSI 技術指標的計算與交易策略回測。策略設計上利用 MACD 判斷趨勢轉折，並用 RSI 過濾動能，搭配交易成本與隔日進場設定，使回測較符合實際交易流程。

整體而言，技術分析策略的價值不只在於是否打敗買進持有，也包含是否能降低回撤、改善風險調整後報酬，或在特定市場環境中提供較穩定的操作規則。策略若要真正應用，仍需要更多樣本外資料與不同市場期間測試。